In [1]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
import zipfile

# ==========================================
# 1. CONFIGURATION
# ==========================================
CONFIG = {
    'img_size': 384,         
    'batch_size': 16,
    'epochs': 2,  # 5
    'lr': 1e-3,              # Higher LR because we are only training the head
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 4,
    'train_dir': '/kaggle/input/automatic-lens-correction/lens-correction-train-cleaned',
    'test_dir': '/kaggle/input/automatic-lens-correction/test-originals',
    # 'output_dir' is no longer needed for file storage, 
    # but we keep the key if you want to reference working directory
    'working_dir': '/kaggle/working/', 
    'model_path': '/kaggle/working/effnet_lens_model.pth'
}

# ==========================================
# 2. DATASET
# ==========================================
class LensDataset(Dataset):
    def __init__(self, root_dir, img_size, is_train=True):
        self.img_size = img_size
        self.is_train = is_train
        self.pairs = []
        if is_train:
            originals = glob.glob(os.path.join(root_dir, "*_original.jpg"))
            for org in originals:
                gen = org.replace("_original.jpg", "_generated.jpg")
                if os.path.exists(gen):
                    self.pairs.append((org, gen))
        else:
            self.pairs = glob.glob(os.path.join(root_dir, "*.jpg"))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.is_train:
            org_path, gen_path = self.pairs[idx]
            img_in = cv2.cvtColor(cv2.imread(org_path), cv2.COLOR_BGR2RGB)
            img_gt = cv2.cvtColor(cv2.imread(gen_path), cv2.COLOR_BGR2RGB)
            
            img_in = cv2.resize(img_in, (self.img_size, self.img_size))
            img_gt = cv2.resize(img_gt, (self.img_size, self.img_size))
            
            # EfficientNet expects normalized data roughly in this range
            img_in = torch.from_numpy(img_in).permute(2, 0, 1).float() / 255.0
            img_gt = torch.from_numpy(img_gt).permute(2, 0, 1).float() / 255.0
            
            # Normalize for ImageNet stats
            normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            return normalize(img_in), img_gt, img_in # Return raw input for warping
        else:
            path = self.pairs[idx]
            img_in = cv2.imread(path)
            h, w = img_in.shape[:2]
            img_in_rgb = cv2.cvtColor(img_in, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_in_rgb, (self.img_size, self.img_size))
            
            img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
            normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            
            filename = os.path.basename(path)
            return normalize(img_tensor), filename, h, w, path

# ==========================================
# 3. EFFICIENTNET MODEL
# ==========================================
class EfficientNetLensModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Load Pretrained EfficientNet
        # We use the 'features' part as the Encoder
        base_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.encoder = base_model.features
        
        # FREEZE THE ENCODER
        for param in self.encoder.parameters():
            param.requires_grad = False
            
        # EfficientNet B0 output channels is 1280 at the final layer
        self.decoder = nn.Sequential(
            # Input: 1280 x 12 x 12 (at 384 input)
            nn.ConvTranspose2d(1280, 512, kernel_size=2, stride=2), 
            nn.BatchNorm2d(512), nn.ReLU(),
            
            nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
            nn.BatchNorm2d(256), nn.ReLU(),
            
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128), nn.ReLU(),
            
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64), nn.ReLU(),
            
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(),
            
            nn.Conv2d(32, 2, kernel_size=3, padding=1) # Output Flow
        )
        
        # Initialize last layer to zero for identity mapping
        nn.init.constant_(self.decoder[-1].weight, 0)
        nn.init.constant_(self.decoder[-1].bias, 0)

    def forward(self, x):
        # Pass through Frozen Encoder
        x = self.encoder(x)
        
        # Pass through Trainable Decoder
        flow = self.decoder(x)
        
        # Final upsample to ensure exact match
        flow = F.interpolate(flow, size=(CONFIG['img_size'], CONFIG['img_size']), mode='bilinear', align_corners=False)
        return flow

def warp_image(x, flow):
    B, C, H, W = x.size()
    xx = torch.linspace(-1.0, 1.0, W).view(1, 1, 1, W).expand(B, -1, H, -1)
    yy = torch.linspace(-1.0, 1.0, H).view(1, 1, H, 1).expand(B, -1, -1, W)
    grid = torch.cat([xx, yy], 1).to(x.device)
    sampling_grid = (grid + flow).permute(0, 2, 3, 1)
    return F.grid_sample(x, sampling_grid, align_corners=True, padding_mode='border')

def apply_high_res_warp(img_path, flow_tensor):
    img_cv = cv2.imread(img_path)
    h_orig, w_orig = img_cv.shape[:2]
    flow_np = flow_tensor.detach().cpu().numpy()[0].transpose(1, 2, 0)
    flow_resized = cv2.resize(flow_np, (w_orig, h_orig), interpolation=cv2.INTER_LINEAR)
    
    grid_x, grid_y = np.meshgrid(np.arange(w_orig), np.arange(h_orig))
    map_x = ((2.0 * grid_x / (w_orig - 1) - 1.0 + flow_resized[:, :, 0] + 1.0) / 2.0) * (w_orig - 1)
    map_y = ((2.0 * grid_y / (h_orig - 1) - 1.0 + flow_resized[:, :, 1] + 1.0) / 2.0) * (h_orig - 1)
    
    return cv2.remap(img_cv, map_x.astype(np.float32), map_y.astype(np.float32), cv2.INTER_LINEAR)

# ==========================================
# 4. TRAINING & INFERENCE
# ==========================================
def train():
    dataset = LensDataset(CONFIG['train_dir'], CONFIG['img_size'], is_train=True)
    dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'])
    
    model = EfficientNetLensModel().to(CONFIG['device'])
    # Only optimize parameters that require grad (The Decoder)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['lr'])
    criterion = nn.L1Loss()
    
    print(f"Training with Frozen EfficientNet Backbone...")
    
    for epoch in range(CONFIG['epochs']):
        model.train()
        loop = tqdm(dataloader)
        for batch_idx, (img_norm, img_gt, img_raw) in enumerate(loop):
            img_norm, img_gt, img_raw = img_norm.to(CONFIG['device']), img_gt.to(CONFIG['device']), img_raw.to(CONFIG['device'])
            
            optimizer.zero_grad()
            flow = model(img_norm) 
            warped = warp_image(img_raw, flow)
            
            loss = criterion(warped, img_gt)
            loss.backward()
            optimizer.step()
            
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())
            
    # Save the model
    torch.save(model.state_dict(), CONFIG['model_path'])
    print(f"Model saved to {CONFIG['model_path']}")
    return model

def inference(model):
    model.eval()
    test_dataset = LensDataset(CONFIG['test_dir'], CONFIG['img_size'], is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    zip_output_path = os.path.join(CONFIG['working_dir'], 'submission_effnet.zip')
    print(f"Running Inference and streaming to {zip_output_path}...")
    
    # Open Zip file directly. Images are written to memory and then to zip,
    # ensuring no individual image files are left on the disk.
    with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        with torch.no_grad():
            for img_norm, filename, h, w, raw_path in tqdm(test_loader):
                img_norm = img_norm.to(CONFIG['device'])
                
                # Get Flow
                flow = model(img_norm)
                
                # Apply correction
                corrected_img = apply_high_res_warp(raw_path[0], flow)
                
                # Encode image to memory buffer (JPG)
                success, buffer = cv2.imencode('.jpg', corrected_img)
                
                if success:
                    # Write buffer directly to zip file
                    # filename is a tuple from dataloader, so use filename[0]
                    zipf.writestr(filename[0], buffer.tobytes())

    print("Inference complete.")
    print(f"Files saved: {CONFIG['model_path']} and {zip_output_path}")

if __name__ == "__main__":
    model = train()
    inference(model)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]


Training with Frozen EfficientNet Backbone...


Epoch 2: 100%|██████████| 1445/1445 [11:32<00:00,  2.09it/s, loss=0.0239]


Model saved to /kaggle/working/effnet_lens_model.pth
Running Inference and streaming to /kaggle/working/submission_effnet.zip...


100%|██████████| 1000/1000 [02:56<00:00,  5.68it/s]

Inference complete.
Files saved: /kaggle/working/effnet_lens_model.pth and /kaggle/working/submission_effnet.zip
